# NeuroVision-X — Kaggle evaluation driver

Thin driver for `scripts/evaluate.py`. Runs sliding-window inference plus postprocessing over a
frozen split and writes per-case metrics, a summary table, and uncropped prediction volumes.

Needs two data sources attached: the preprocessed dataset, and a dataset containing the
checkpoint to score. Both are **discovered**, not hardcoded — Kaggle mounts sources one level
deeper than documented.

Unlike training, this is cheap: ~1.3 s/case, so a 189-case split is ~15 min on a T4.

## 1. Session config — the only cell you edit

In [ ]:
REPO_URL   = "https://github.com/AmishhYadav/NeuroVision-X.git"
GIT_REF    = "main"
SPLIT      = "test"          # the untouched split; "val" only for iterating
EXPERIMENT = "baseline_unet3d"
CKPT_NAME  = "best.pt"       # filename to look for in the attached checkpoint dataset
OVERRIDES  = ["model=unet3d", "data.num_workers=2"]

## 2. Code + dependencies

Install list is derived from `requirements.txt` minus its own `# kaggle-exclude:` line —
installing our pinned numpy over Kaggle's breaks its ABI-linked scipy. `sys.path` rather than
`pip install -e`, because Kaggle runs Python 3.12 and `pyproject.toml` pins `<3.12`.

In [ ]:
!git clone -q --depth 1 -b {GIT_REF} {REPO_URL} /kaggle/working/repo
import pathlib
import re
import subprocess
import sys

_req = pathlib.Path("/kaggle/working/repo/requirements.txt").read_text()
_excl = {w for m in re.findall(r"^#\s*kaggle-exclude:\s*(.+)$", _req, re.M) for w in m.split()}
_keep = [
    ln for ln in _req.splitlines()
    if ln.strip() and not ln.lstrip().startswith("#")
    and re.split(r"[=<>~!\[]", ln.strip())[0].strip() not in _excl
]
print("installing:", " ".join(_keep))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_keep], check=True)

In [ ]:
import torch

sys.path.insert(0, "/kaggle/working/repo/src")
sys.path.insert(0, "/kaggle/working/repo/scripts")
import neurovision  # noqa: F401
import scipy.ndimage  # noqa: F401 -- canary: breaks if our numpy pin overwrote Kaggle's

assert torch.cuda.is_available(), "No CUDA: enable the GPU accelerator."
_cap = "sm_%d%d" % torch.cuda.get_device_capability(0)
# is_available() is not sufficient -- a Kaggle P100 (sm_60) reports True while
# every kernel launch fails against this torch build. Execute something.
try:
    (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
except Exception as exc:
    raise RuntimeError(f"{_cap} cannot run a kernel: {exc}. Use machine_shape=NvidiaTeslaT4.") from exc
print(f"{torch.cuda.get_device_name(0)}  {_cap}  torch {torch.__version__}  python {sys.version.split()[0]}")

## 3. Discover the dataset and the checkpoint

Both raise immediately, listing what *is* mounted, rather than failing later.

In [ ]:
from pathlib import Path

_pats = ("splits.yaml", "*/splits.yaml", "*/*/splits.yaml", "*/*/*/splits.yaml")
_hits = sorted({p.parent for pat in _pats for p in Path("/kaggle/input").glob(pat)
                if (p.parent / "preprocessed").is_dir()})
if len(_hits) != 1:
    raise FileNotFoundError(f"Need exactly one preprocessed dataset, found {[str(h) for h in _hits]}")
DATA = _hits[0]
PREP, SPLITS = DATA / "preprocessed", DATA / "splits.yaml"

_ck = sorted(set(Path("/kaggle/input").glob(f"**/{CKPT_NAME}")))
if len(_ck) != 1:
    _tree = sorted(str(q.relative_to("/kaggle/input")) for q in Path("/kaggle/input").glob("*/*"))
    raise FileNotFoundError(
        f"Need exactly one {CKPT_NAME} under /kaggle/input, found {[str(h) for h in _ck]}. "
        f"Mounted: {_tree[:20]}"
    )
CKPT = _ck[0]
_meta = torch.load(CKPT, weights_only=True, map_location="cpu")
print(f"data       : {PREP}")
print(f"checkpoint : {CKPT}")
print(f"  epoch={_meta['epoch']}  {_meta['best_metric_name']}={_meta['best_metric']:.4f}")
del _meta

## 4. Evaluate

`strict_arch_check` (on by default) refuses to run if the checkpoint's stored `model.name`
disagrees with the current config — `load_state_dict` alone would not catch every mismatch.

In [ ]:
import hydra

from evaluate import run_evaluation

OUT = f"/kaggle/working/eval_{SPLIT}"
overrides = [
    f"data.root_dir={DATA}", f"data.preprocessing.out_dir={PREP}", f"data.splits.path={SPLITS}",
    f"experiment_name={EXPERIMENT}",
    f"inference.evaluation.split={SPLIT}",
    f"inference.evaluation.checkpoint={CKPT}",
    f"inference.evaluation.out_dir={OUT}",
    "wandb.mode=disabled",
    *OVERRIDES,
]
with hydra.initialize_config_dir(version_base="1.3", config_dir="/kaggle/working/repo/configs"):
    cfg = hydra.compose("config", overrides=overrides)
per_case = run_evaluation(cfg)

## 5. Results

`per_case_metrics.csv` keeps the `gt_empty_<region>` flags, so the ignore-empty convention can be
recomputed later without re-running inference. `n_missing` counts cases where HD95 was undefined
(one side empty) rather than silently dropping them.

In [ ]:
import pandas as pd

summary = pd.read_csv(f"{OUT}/summary.csv", index_col=0)
cols = [c for c in ("mean", "std", "median", "count", "n_missing") if c in summary.columns]
rows = [r for r in summary.index if r.startswith(("dice_", "iou_", "hd95_", "gt_empty_"))]
print(summary.loc[rows, cols].to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\ncases scored: {len(per_case)}")
print(f"predictions : {len(list(pathlib.Path(OUT + '/predictions').glob('*.npy')))} files")